# Lesson 9.1 - Deep Reinforcement Learning (toy Gym environment demo)

## Objectives

- Map MDP concepts to executable code.
- Train a simple tabular Q-learning agent in FrozenLake.
- Understand reward curves and convergence behavior.


In [ ]:
from __future__ import annotations

import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

np.random.seed(42)


## Environment Setup

FrozenLake is a small discrete environment, ideal for showing tabular RL mechanics.

- state space: finite and discrete
- action space: finite and discrete
- sparse reward signal


In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=True)

n_states = env.observation_space.n
n_actions = env.action_space.n

print(f"states={n_states}, actions={n_actions}")


## Q-Learning Agent

Update rule:

\[
Q(s,a) \leftarrow Q(s,a)+lpha [r+\gamma \max_{a'}Q(s',a')-Q(s,a)]
\]


In [ ]:
q_table = np.zeros((n_states, n_actions), dtype=np.float32)

alpha = 0.1
gamma = 0.99
epsilon_start = 1.0
epsilon_end = 0.05
epsilon_decay = 0.9995

episodes = 4000
max_steps = 100

rewards = []
epsilon = epsilon_start

for ep in range(episodes):
    state, _ = env.reset(seed=42 + ep)
    total_reward = 0.0

    for _ in range(max_steps):
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(q_table[state]))

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        best_next = np.max(q_table[next_state])
        td_target = reward + gamma * best_next * (0 if done else 1)
        td_error = td_target - q_table[state, action]
        q_table[state, action] += alpha * td_error

        state = next_state
        total_reward += reward
        if done:
            break

    epsilon = max(epsilon_end, epsilon * epsilon_decay)
    rewards.append(total_reward)

print("Training complete")


## Reward Curve


In [ ]:
def moving_average(x, w=100):
    x = np.array(x)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w)/w, mode="valid")

ma = moving_average(rewards, 200)

plt.figure(figsize=(10, 4))
plt.plot(ma)
plt.title("Moving Average Episode Reward (window=200)")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.grid(True, alpha=0.3)
plt.show()


## Evaluate Learned Policy


In [ ]:
def evaluate_policy(env_name: str, q_table: np.ndarray, episodes: int = 200) -> float:
    test_env = gym.make(env_name, is_slippery=True)
    returns = []
    for ep in range(episodes):
        s, _ = test_env.reset(seed=1000 + ep)
        done = False
        total = 0.0
        while not done:
            a = int(np.argmax(q_table[s]))
            s, r, term, trunc, _ = test_env.step(a)
            done = term or trunc
            total += r
        returns.append(total)
    return float(np.mean(returns))

avg_return = evaluate_policy("FrozenLake-v1", q_table)
print(f"Average evaluation return over 200 episodes: {avg_return:.3f}")


## Optional SB3 Extension

If `stable-baselines3` is installed, you can compare tabular Q-learning with PPO or DQN.


In [ ]:
# Optional (commented) example:
# from stable_baselines3 import DQN
# sb3_env = gym.make("CartPole-v1")
# model = DQN("MlpPolicy", sb3_env, verbose=0)
# model.learn(total_timesteps=20_000)


## Connect to Theory

- `env` defines the MDP interface.
- Q-table approximates action-value function \(Q(s,a)\) in tabular form.
- Epsilon-greedy controls exploration vs exploitation.
- Reward curve approximates policy improvement over training.


## Business Case Studies & Exceptions

### Case Study A: Inventory Replenishment Control

RL can optimize reorder policies under uncertain demand by learning long-term cost trade-offs, but only when a reliable simulator or safe online feedback loop exists.

### Case Study B: Dynamic Pricing

RL can adapt prices over time to maximize margin and conversion. In regulated sectors, constraints (fairness, anti-price-gouging rules) must be hard-coded.

### Exceptions

- If decisions are one-step and context-based, contextual bandits or supervised models are often simpler.
- If reward definitions are weak, RL can optimize the wrong behavior despite improving returns.


## Interview Questions & Answers

1. **What is an MDP in RL?**  
   Formal environment model with states, actions, transitions, rewards, and discount factor.
2. **What does Q-learning estimate?**  
   Optimal action-value function \(Q^*(s,a)\).
3. **Why use epsilon-greedy?**  
   To force exploration while gradually exploiting learned values.
4. **What is sparse reward?**  
   Reward appears infrequently, making credit assignment hard.
5. **Why is FrozenLake hard?**  
   Stochastic transitions + sparse rewards.
6. **How do you know training is working?**  
   Moving-average return should increase and stabilize.
7. **What is sample inefficiency?**  
   Requiring many interactions to learn useful policies.
8. **When move from tabular to deep RL?**  
   When state/action spaces are too large for tables.
9. **Why are target networks useful in DQN?**  
   They stabilize bootstrapped targets.
10. **When should RL be avoided?**  
   When problem is static, one-step, or lacks safe reward signal.
